# Health Facility Data Quality Assessment & Reporting
## Rumphi District, Malawi — Synthetic Portfolio Dataset

**Author:** Chico Mkandawire
**Purpose:** Portfolio project demonstrating data quality assessment, validation, and reporting skills relevant to Data Technician, Data Analyst, and Monitoring & Evaluation (M&E) roles in the health sector.

---

**Important — synthetic data notice**

Every facility name, reported figure, and finding in this notebook comes from data **generated programmatically in Section 1 below**, using a fixed random seed. Nothing here represents actual Ministry of Health, DHIS2, or Rumphi District data, and no real patient information is used anywhere in this project. Facility names are illustrative and were chosen to be plausible for the district, not sourced from any real facility list or dataset.

**Why build this on synthetic data at all:** the goal of this project is to demonstrate a *data quality assessment workflow* — the kind of thing a Data Technician or M&E Assistant does every reporting cycle: checking completeness, reconciling multiple data sources, flagging late or unusual reports, and turning that into a prioritized follow-up list. That workflow is what's being demonstrated, not a claim about any real district's health performance. The synthetic generator deliberately injects realistic problems (missing values, source disagreements, outliers, late reports) so there's something genuine to detect — every number reported later in this notebook is a real calculation on this generated data, not an invented conclusion.


## 1. Generate the Synthetic Dataset

Eight fictional facilities, five programme areas (each with two indicators), twelve months of 2025, and three reporting sources (Paper Register, EMR, DHIS2) that are expected to broadly agree but don't always.

A fixed `np.random.seed(42)` makes this reproducible — running this notebook again produces the exact same dataset and therefore the exact same downstream numbers.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys
import json

# Resolve the project root whether the notebook is launched from the repo root
# or from the notebooks/ directory.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "src"))

from data_generator import HealthDataGenerator
from data_quality import DataQualityAssessor

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110

In [ ]:
# Use the new data generator module
generator = HealthDataGenerator(random_seed=42)

print(f"Facilities: {len(generator.facilities)}")
print(f"Programme areas: {list(generator.programme_indicators.keys())}")
print(f"Months: {generator.months.min().date()} to {generator.months.max().date()} ({len(generator.months)} months)")

In [ ]:
# Generate synthetic data with ground truth tracking
df, injection_ledger = generator.generate()

print(f"Generated dataset: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"\nInjection summary:")
print(f"  Missing values: {len(injection_ledger['missing_values']):,}")
print(f"  Source discrepancies: {len(injection_ledger['source_discrepancies']):,}")
print(f"  Outliers: {len(injection_ledger['outliers']):,}")
print(f"  Late reports: {len(injection_ledger['late_reports']):,}")

df.head()

In [ ]:
# Save raw data and injection ledger
RAW_DIR = PROJECT_ROOT / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

df.to_csv(RAW_DIR / "synthetic_health_facility_data.csv", index=False)
generator.save_injection_ledger(RAW_DIR / "injection_ledger.json")

print(f"Raw synthetic dataset saved: {df.shape[0]:,} rows -> {RAW_DIR / 'synthetic_health_facility_data.csv'}")
print(f"Injection ledger saved: {RAW_DIR / 'injection_ledger.json'}")

## 2. Initial Inspection and Validation

In [ ]:
# Initialize data quality assessor
assessor = DataQualityAssessor(df)

# Validate schema
validation_results = assessor.validate_schema()
print("Schema Validation:")
print(f"  Has required columns: {validation_results['has_required_columns']}")
print(f"  Total rows: {validation_results['total_rows']:,}")
print(f"  Duplicate keys: {validation_results['duplicate_keys']}")
print(f"  Invalid dates: {validation_results['invalid_dates']}")
print(f"  Date range: {validation_results['date_range'][0]} to {validation_results['date_range'][1]}")

print("\nRows:", df.shape[0])
print("Columns:", df.shape[1])
print()
df.info()

In [ ]:
df.describe(include="all").T

In [ ]:
missing = df.isnull().sum().sort_values(ascending=False)
missing[missing > 0]

The missing values here are in `Reported_Value` — exactly where the injection placed them. This confirms the detection step is finding the real injected problem, not something spurious.

## 3. Data Cleaning and Quality Calculations

In [ ]:
# Convert Month to datetime and coerce numeric columns
df["Month"] = pd.to_datetime(df["Month"])

numeric_columns = ["Reported_Value", "Target_Value", "Paper_Value", "EMR_Value",
                    "DHIS2_Value", "Completeness_Score"]
for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")

df["Completeness_Score"] = df["Completeness_Score"].clip(0, 1)

# Add missing value flag
df["Reported_Value_Missing"] = df["Reported_Value"].isna()

print(f"Rows flagged as missing a reported value: {df['Reported_Value_Missing'].sum():,} "
      f"({df['Reported_Value_Missing'].mean():.1%} of all records)")

### 3.1 Calculate Completeness from Actual Fields

The original `Completeness_Score` was randomly assigned. Now we calculate a defensible completeness measure from the actual presence of required fields.

In [ ]:
# Calculate completeness from actual field presence
assessor = DataQualityAssessor(df)
df = assessor.calculate_field_completeness()

print(f"Average calculated completeness: {df['Calculated_Completeness'].mean():.1%}")
print(f"Average simulated completeness: {df['Completeness_Score'].mean():.1%}")
print(f"\nComparison (first 10 rows):")
print(df[['Completeness_Score', 'Calculated_Completeness']].head(10))

### 3.2 Calculate Timeliness from Dates

The original `Timeliness` field was randomly assigned. For a real operational workflow, timeliness would be calculated from submission dates and deadlines. Since this synthetic dataset doesn't have actual submission/deadline dates, we'll use the simulated field as a placeholder and add a calculated field that would work with real dates.

In [ ]:
# For this synthetic dataset without actual submission/deadline dates, use the
# generator's simulated Timeliness field as a clearly labelled placeholder.
# In an operational system, calculate this from submission and reporting-deadline dates.
df = assessor.calculate_timeliness_from_dates()
print(f"Timely reporting rate: {(df['Calculated_Timeliness'] == 'On Time').mean():.1%}")


## 4. Data Validation — Source Consistency

In [ ]:
# Assess source consistency with both baseline and improved rules
df = assessor.assess_source_consistency(
    baseline_threshold=0.10,
    improved_threshold=0.20,
    absolute_tolerance=5
)

print("Baseline 10% rule:")
print(df["Consistency_Flag_Baseline"].value_counts())
print(f"\nImproved rule (20% relative + 5 absolute for low volumes):")
print(df["Consistency_Flag_Improved"].value_counts())

# Use the volume-aware hybrid rule for downstream analysis.
df["Consistency_Flag"] = df["Consistency_Flag_Improved"]

The baseline 10% rule flags many more records than the 5% discrepancy injection rate. This is expected: natural random variation between sources frequently exceeds 10% for low-volume indicators. The improved rule (20% relative + 5 absolute) is more appropriate for operational use.

## 5. Data Quality Assessment — Completeness

### 5.1 By facility (using calculated completeness)

In [ ]:
facility_completeness = df.groupby("Facility")["Calculated_Completeness"].mean().sort_values()
facility_completeness

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
facility_completeness.plot(kind="barh", ax=ax, color="#4C72B0")
ax.set_xlabel("Average Calculated Completeness")
ax.set_ylabel("Facility")
ax.set_title("Average Data Completeness by Facility (Calculated from Field Presence)")
plt.tight_layout()
plt.savefig(PROJECT_ROOT / "visuals" / "charts" / "01_completeness_by_facility.png", dpi=140, bbox_inches="tight")
plt.show()

### 5.2 By programme area (using calculated completeness)

In [ ]:
programme_completeness = df.groupby("Programme")["Calculated_Completeness"].mean().sort_values()
programme_completeness

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
programme_completeness.plot(kind="barh", ax=ax, color="#55A868")
ax.set_xlabel("Average Calculated Completeness")
ax.set_ylabel("Programme Area")
ax.set_title("Average Data Completeness by Programme Area (Calculated from Field Presence)")
plt.tight_layout()
plt.savefig(PROJECT_ROOT / "visuals" / "charts" / "02_completeness_by_programme.png", dpi=140, bbox_inches="tight")
plt.show()

## 6. Data Quality Assessment — Timeliness

In [ ]:
timeliness_by_facility = df.groupby("Facility")["Calculated_Timeliness"].apply(lambda x: (x == "On Time").mean()).sort_values()
timeliness_by_facility

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
timeliness_by_facility.plot(kind="barh", ax=ax, color="#C44E52")
ax.set_xlabel("Proportion On Time")
ax.set_ylabel("Facility")
ax.set_title("Timeliness by Facility (Proportion of Reports Submitted On Time)")
plt.tight_layout()
plt.savefig(PROJECT_ROOT / "visuals" / "charts" / "03_timeliness_by_facility.png", dpi=140, bbox_inches="tight")
plt.show()

## 7. Data Quality Assessment — Source Consistency

In [ ]:
consistency_by_facility = df.groupby("Facility")["Consistency_Flag"].apply(lambda x: (x == "Consistent").mean()).sort_values()
consistency_by_facility

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
consistency_by_facility.plot(kind="barh", ax=ax, color="#8172B2")
ax.set_xlabel("Proportion Consistent")
ax.set_ylabel("Facility")
ax.set_title("Source Consistency by Facility (Hybrid 20% Relative + 5-Unit Absolute Threshold)")
plt.tight_layout()
plt.savefig(PROJECT_ROOT / "visuals" / "charts" / "04_consistency_by_facility.png", dpi=140, bbox_inches="tight")
plt.show()

## 8. Validation of Injected Issues Against Detection Rules

In [ ]:
# Validate that injected issues are detected by the rules
validation_results = []

# Check missing values
missing_detected = df[df["Reported_Value_Missing"] == True]
validation_results.append({
    "Issue": "Missing Value",
    "Injected Count": df["Reported_Value_Missing"].sum(),
    "Detected": len(missing_detected),
    "Detection Rate": len(missing_detected) / df["Reported_Value_Missing"].sum() if df["Reported_Value_Missing"].sum() > 0 else 0
})

# Check source consistency (baseline 10% rule)
inconsistent_records = df[df["Consistency_Flag_Baseline"] == "Inconsistent"]
validation_results.append({
    "Issue": "Source Inconsistency (10% rule)",
    "Expected Rate": "~5% (injected)",
    "Detected Count": len(inconsistent_records),
    "Detection Rate": len(inconsistent_records) / len(df)
})

# Check timeliness
late_records = df[df["Calculated_Timeliness"] == "Late"]
validation_results.append({
    "Issue": "Late Reporting",
    "Expected Rate": "~15% (injected)",
    "Detected Count": len(late_records),
    "Detection Rate": len(late_records) / len(df)
})

validation_df = pd.DataFrame(validation_results)
print("Validation of Injected Issues Against Detection Rules:")
print(validation_df.to_string(index=False))

The source inconsistency detection rate is higher than the 5% injection rate due to natural random variation between sources exceeding the 10% threshold. This is expected behavior and demonstrates why the improved rule (20% relative + 5 absolute) is more appropriate for operational use.

## 9. Outlier Detection

In [ ]:
# Detect outliers using IQR method, grouped by indicator to account for different scales
# This is more appropriate than a global threshold since indicators have different ranges

def detect_outliers_iqr(group, multiplier=1.5):
    """Detect outliers using IQR method for a single group."""
    values = group["Reported_Value"].dropna()
    if len(values) < 4:  # Need minimum data for IQR
        return pd.Series([False] * len(group), index=group.index)
    
    q1 = values.quantile(0.25)
    q3 = values.quantile(0.75)
    iqr = q3 - q1
    
    if iqr == 0:  # All values are the same
        return pd.Series([False] * len(group), index=group.index)
    
    lower_bound = q1 - multiplier * iqr
    upper_bound = q3 + multiplier * iqr
    
    outliers = (group["Reported_Value"] < lower_bound) | (group["Reported_Value"] > upper_bound)
    return outliers

# Apply outlier detection by indicator
df["Outlier_Flag"] = df.groupby("Indicator").apply(detect_outliers_iqr).reset_index(level=0, drop=True)

print(f"Outliers detected: {df['Outlier_Flag'].sum()} ({df['Outlier_Flag'].mean():.1%} of records)")
print(f"\nOutliers by indicator:")
print(df[df["Outlier_Flag"]].groupby("Indicator").size())

The IQR method grouped by indicator is more appropriate than a global threshold because different indicators have different scales and distributions. This reduces false positives for indicators with naturally high variance.

## 10. Facility Risk Score

In [ ]:
# Calculate facility risk scores using rates (not raw counts)
# This normalizes by facility size and reporting volume

def calculate_facility_risk_rates(df):
    """Calculate risk score for each facility using issue rates."""
    facility_metrics = []
    
    for facility in df["Facility"].unique():
        facility_data = df[df["Facility"] == facility]
        total_records = len(facility_data)
        
        if total_records == 0:
            continue
        
        # Calculate issue rates
        missing_rate = facility_data["Reported_Value_Missing"].mean()
        inconsistent_rate = (facility_data["Consistency_Flag"] == "Review Required").mean()
        late_rate = (facility_data["Calculated_Timeliness"] == "Late").mean()
        outlier_rate = facility_data["Outlier_Flag"].mean()
        
        # Average completeness (lower is worse)
        avg_completeness = facility_data["Calculated_Completeness"].mean()
        
        # Calculate risk score (0-100, higher = worse)
        # Each dimension contributes 0-20 points
        risk_score = (
            missing_rate * 20 +
            inconsistent_rate * 20 +
            late_rate * 20 +
            outlier_rate * 20 +
            (1 - avg_completeness) * 20
        )
        
        facility_metrics.append({
            "Facility": facility,
            "Total_Records": total_records,
            "Missing_Rate": missing_rate,
            "Inconsistent_Rate": inconsistent_rate,
            "Late_Rate": late_rate,
            "Outlier_Rate": outlier_rate,
            "Avg_Completeness": avg_completeness,
            "Risk_Score": risk_score
        })
    
    return pd.DataFrame(facility_metrics).sort_values("Risk_Score", ascending=False)

facility_risk_df = calculate_facility_risk_rates(df)
print("Facility Risk Scores (using rates, normalized by volume):")
print(facility_risk_df[["Facility", "Risk_Score", "Missing_Rate", "Inconsistent_Rate", "Late_Rate", "Outlier_Rate", "Avg_Completeness"]].to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
colors = ["#C44E52" if score > 50 else "#FBC15E" if score > 30 else "#55A868" for score in facility_risk_df["Risk_Score"]]
facility_risk_df.plot(kind="barh", x="Facility", y="Risk_Score", ax=ax, color=colors, legend=False)
ax.set_xlabel("Risk Score (0-100, higher = worse)")
ax.set_ylabel("Facility")
ax.set_title("Facility Risk Scores (Normalized by Reporting Volume)")
ax.axvline(x=50, color="#C44E52", linestyle="--", alpha=0.5, label="High Risk")
ax.axvline(x=30, color="#FBC15E", linestyle="--", alpha=0.5, label="Medium Risk")
ax.legend()
plt.tight_layout()
plt.savefig(PROJECT_ROOT / "visuals" / "charts" / "05_facility_risk_scores.png", dpi=140, bbox_inches="tight")
plt.show()

## 11. Trend Analysis

In [ ]:
# Analyze trends over time using proper aggregation
# Aggregate by month and calculate rates, not raw counts.
df["Month"] = pd.to_datetime(df["Month"]).dt.to_period("M")

# Calculate monthly metrics using rates.
monthly_metrics = df.groupby("Month").agg({
    "Reported_Value_Missing": "mean",
    "Consistency_Flag": lambda x: (x == "Review Required").mean(),
    "Calculated_Timeliness": lambda x: (x == "Late").mean(),
    "Outlier_Flag": "mean",
    "Calculated_Completeness": "mean",
}).reset_index()
monthly_metrics["Month"] = monthly_metrics["Month"].astype(str)
print("Monthly Data Quality Trends (using rates):")
print(monthly_metrics)


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Missing rate trend
axes[0, 0].plot(monthly_metrics["Month"], monthly_metrics["Reported_Value_Missing"], marker="o", color="#C44E52")
axes[0, 0].set_title("Missing Value Rate Over Time")
axes[0, 0].set_ylabel("Rate")
axes[0, 0].tick_params(axis="x", rotation=45)

# Inconsistency rate trend
axes[0, 1].plot(monthly_metrics["Month"], monthly_metrics["Consistency_Flag"], marker="o", color="#8172B2")
axes[0, 1].set_title("Source Inconsistency Rate Over Time")
axes[0, 1].set_ylabel("Rate")
axes[0, 1].tick_params(axis="x", rotation=45)

# Late reporting rate trend
axes[1, 0].plot(monthly_metrics["Month"], monthly_metrics["Calculated_Timeliness"], marker="o", color="#FBC15E")
axes[1, 0].set_title("Late Reporting Rate Over Time")
axes[1, 0].set_ylabel("Rate")
axes[1, 0].tick_params(axis="x", rotation=45)

# Completeness trend
axes[1, 1].plot(monthly_metrics["Month"], monthly_metrics["Calculated_Completeness"], marker="o", color="#4C72B0")
axes[1, 1].set_title("Average Completeness Over Time")
axes[1, 1].set_ylabel("Completeness (0-1)")
axes[1, 1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.savefig(PROJECT_ROOT / "visuals" / "charts" / "06_trend_analysis.png", dpi=140, bbox_inches="tight")
plt.show()

## 12. Export Processed Data

In [ ]:
# Export the processed dataset with all calculated metrics
output_path = PROJECT_ROOT / "data" / "processed" / "health_facility_data_processed.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(output_path, index=False)
print(f"Processed data exported to: {output_path}")
print(f"Total records: {len(df)}")
print(f"Columns: {len(df.columns)}")